## ETAPA 7 - Guardado del modelo final en formatos .joblib y .pkl 
### Objetivo 
En este notebook se guardará el modelo final del proyecto, correspondiente al Random Forest ajustado, con el fin de reutilizarlo posteriormente sin necesidad de reentrenamiento. Así mismo para garantizar la funcionalidad se reconstruirá el flujo mínimo de preparación de datos, se entrenará nuevamente el modelo final seleccionado y se guardará en formatos `.joblib` y `.pkl`, con el fin de asegurar su reutilización futura sin depender de variables temporales del entorno. 

### Regla de trabajo 
El notebook será autosuficiente. Por ello, volverá a cargar el dataset featured, reconstruirá las variables predictoras, dividirá los datos de forma estratificada, entrenará el Random Forest ajustado final y luego lo serializará en ambos formatos.

In [1]:
# ==================================================
# FASE 7 - GUARDADO AUTOSUFICIENTE DEL MODELO FINAL
# RISK SCORE OSCE
# EN FORMATOS .JOBLIB Y .PKL
# ==================================================

import os
import joblib
import pickle
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ------------------------------------------
# 1. Definir rutas
# ------------------------------------------

ruta_featured = r"C:\Proyecto-Risk-Score-OECE\data\processed\CONOSCE_ADJUDICACIONES_2018_2026_CONSOLIDADO_FEATURED.csv"

ruta_modelos = r"C:\Proyecto-Risk-Score-OECE\models"
os.makedirs(ruta_modelos, exist_ok=True)

ruta_joblib = os.path.join(ruta_modelos, "riskscore_osce_random_forest.joblib")
ruta_pkl = os.path.join(ruta_modelos, "riskscore_osce_random_forest.pkl")
ruta_artefacto = os.path.join(ruta_modelos, "artefacto_riskscore_osce_random_forest.joblib")
ruta_columnas = os.path.join(ruta_modelos, "columnas_entrenamiento_riskscore_osce.csv")

In [2]:
# ------------------------------------------
# 2. Cargar dataset
# ------------------------------------------

df_featured = pd.read_csv(ruta_featured, sep=";")
df_modelo = df_featured.copy()

print("Dimensión del dataset featured:", df_modelo.shape)

print("\nColumnas disponibles:")
print(df_modelo.columns.tolist())

display(df_modelo.head())

Dimensión del dataset featured: (3875, 42)

Columnas disponibles:
['RUC', 'NOMBRE_RAZONODENOMINACIONSOCIAL', 'FECHA_INICIO', 'FECHA_FIN', 'NUMERO_RESOLUCION', 'ID_MOTIVO_INFRACCION', 'DE_MOTIVO_INFRACCION', 'motivo_grupo', 'duracion_dias', 'duracion_grupo', 'ruc_prefijo', 'fecha_segunda', 'target_sancionado_24m', 'FECHA_INICIO_DT', 'FECHA_FIN_DT', 'ANIO_INICIO_SANCION', 'MES_INICIO_SANCION', 'TRIMESTRE_INICIO_SANCION', 'PERIODO_INICIO_SANCION', 'DURACION_MESES', 'DURACION_ANIOS', 'SANCION_MAYOR_1_ANIO', 'SANCION_MAYOR_3_ANIOS', 'LOG_DURACION_DIAS', 'DURACION_DIAS_CORREGIDA', 'RUC_STR', 'RUC_PREFIJO', 'ES_PERSONA_NATURAL', 'ES_PERSONA_JURIDICA', 'ES_OTRO_TIPO_RUC', 'MOTIVO_TEXTO', 'MOTIVO_DOC_FALSA', 'MOTIVO_INFO_INEXACTA', 'MOTIVO_INCUMPLIMIENTO', 'MOTIVO_CONTRATAR_IMPEDIDO', 'NUM_MOTIVOS_INFRACCION', 'TIENE_MULTIPLES_MOTIVOS', 'TOTAL_SANCIONES_PROVEEDOR', 'ORDEN_SANCION_PROVEEDOR', 'PROVEEDOR_CON_MULTIPLES_SANCIONES', 'FECHA_PRIMERA_SANCION_PROVEEDOR', 'DIAS_DESDE_PRIMERA_SANCION']


,RUC,NOMBRE_RAZONODENOMINACIONSOCIAL,FECHA_INICIO,FECHA_FIN,NUMERO_RESOLUCION,ID_MOTIVO_INFRACCION,DE_MOTIVO_INFRACCION,motivo_grupo,duracion_dias,duracion_grupo,...,MOTIVO_INFO_INEXACTA,MOTIVO_INCUMPLIMIENTO,MOTIVO_CONTRATAR_IMPEDIDO,NUM_MOTIVOS_INFRACCION,TIENE_MULTIPLES_MOTIVOS,TOTAL_SANCIONES_PROVEEDOR,ORDEN_SANCION_PROVEEDOR,PROVEEDOR_CON_MULTIPLES_SANCIONES,FECHA_PRIMERA_SANCION_PROVEEDOR,DIAS_DESDE_PRIMERA_SANCION
0,2029124996,G & D CORPORACION DE NEGOCIOS LACTEOS S.A. COR...,20050608,NaN,508-2005-TC-SU,8,DOCUMENTOS FALSOS,Documentación falsa/inexacta,NaN,Sin fecha fin,...,0,0,0,1,0,1,1,0,2005-06-08,0
1,10000282125,GOMEZ CASTRO JUANA,20200616,20230716.0,1110-2020-TCE-S2,"215,",Presentar documentos falsos o adulterados a la...,Documentación falsa/inexacta,1125.0,Más de 3 años,...,0,0,0,1,0,1,1,0,2020-06-16,0
2,10000710623,OLIVEIRA DE MACHUCA LITA,20191107,20230207.0,2903-2019-TCE-S2,"214,215,",Presentar información inexacta a las Entidades...,Documentación falsa/inexacta,1188.0,Más de 3 años,...,1,0,0,2,1,1,1,0,2019-11-07,0
3,10001253609,AGUIRRE VARGAS GUSTAVO ALFREDO,20190807,20220907.0,2147-2019-TCE-S3,"214,215,",Presentar información inexacta a las Entidades...,Documentación falsa/inexacta,1127.0,Más de 3 años,...,1,0,0,2,1,1,1,0,2019-08-07,0
4,10002101756,MENDOZA PENA JULIO CESAR,20231005,20240205.0,3820-2023-TCE-S6,"241,",f) Ocasionar que la Entidad resuelva el contra...,Resolución/rescisión contractual,123.0,Hasta 6 meses,...,0,0,0,1,0,1,1,0,2023-10-05,0


In [3]:
# ------------------------------------------
# 3. Definir target y variables predictoras
# ------------------------------------------

target = "target_sancionado_24m"

columnas_modelo = [
    # Variables temporales
    "ANIO_INICIO_SANCION",
    "MES_INICIO_SANCION",
    "TRIMESTRE_INICIO_SANCION",

    # Variables de duración / severidad
    "DURACION_MESES",
    "DURACION_ANIOS",
    "SANCION_MAYOR_1_ANIO",
    "SANCION_MAYOR_3_ANIOS",
    "LOG_DURACION_DIAS",

    # Variables del tipo de proveedor
    "ES_PERSONA_NATURAL",
    "ES_PERSONA_JURIDICA",
    "ES_OTRO_TIPO_RUC",

    # Variables del motivo de infracción
    "MOTIVO_DOC_FALSA",
    "MOTIVO_INFO_INEXACTA",
    "MOTIVO_INCUMPLIMIENTO",
    "MOTIVO_CONTRATAR_IMPEDIDO",
    "NUM_MOTIVOS_INFRACCION",
    "TIENE_MULTIPLES_MOTIVOS",

    # Variables categóricas
    "motivo_grupo",
    "duracion_grupo"
]

# Verificar columnas existentes
columnas_existentes = [col for col in columnas_modelo if col in df_modelo.columns]
columnas_faltantes = [col for col in columnas_modelo if col not in df_modelo.columns]

print("Columnas usadas en el modelo:")
print(columnas_existentes)

print("\nColumnas faltantes:")
print(columnas_faltantes)

df_modelo = df_modelo[columnas_existentes + [target]].copy()

X = df_modelo.drop(columns=[target])
y = df_modelo[target]

print("\nDistribución de clases en la variable objetivo:")
print(y.value_counts())

print("\nPorcentaje por clase:")
print((y.value_counts(normalize=True) * 100).round(2))

Columnas usadas en el modelo:
['ANIO_INICIO_SANCION', 'MES_INICIO_SANCION', 'TRIMESTRE_INICIO_SANCION', 'DURACION_MESES', 'DURACION_ANIOS', 'SANCION_MAYOR_1_ANIO', 'SANCION_MAYOR_3_ANIOS', 'LOG_DURACION_DIAS', 'ES_PERSONA_NATURAL', 'ES_PERSONA_JURIDICA', 'ES_OTRO_TIPO_RUC', 'MOTIVO_DOC_FALSA', 'MOTIVO_INFO_INEXACTA', 'MOTIVO_INCUMPLIMIENTO', 'MOTIVO_CONTRATAR_IMPEDIDO', 'NUM_MOTIVOS_INFRACCION', 'TIENE_MULTIPLES_MOTIVOS', 'motivo_grupo', 'duracion_grupo']

Columnas faltantes:
[]

Distribución de clases en la variable objetivo:
target_sancionado_24m
0    3139
1     736
Name: count, dtype: int64

Porcentaje por clase:
target_sancionado_24m
0    81.01
1    18.99
Name: proportion, dtype: float64


In [4]:
# ------------------------------------------
# 4. Limpieza básica antes del entrenamiento
# ------------------------------------------

X = X.replace([np.inf, -np.inf], np.nan)

columnas_numericas = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
columnas_categoricas = X.select_dtypes(include=["object"]).columns.tolist()

# Guardaremos estas reglas en el artefacto final
medianas_numericas = {}

for col in columnas_numericas:
    mediana = X[col].median()
    medianas_numericas[col] = mediana
    X[col] = X[col].fillna(mediana)

for col in columnas_categoricas:
    X[col] = X[col].fillna("SIN_DATO")

print("Nulos después del tratamiento:")
print(X.isnull().sum())

Nulos después del tratamiento:
ANIO_INICIO_SANCION          0
MES_INICIO_SANCION           0
TRIMESTRE_INICIO_SANCION     0
DURACION_MESES               0
DURACION_ANIOS               0
SANCION_MAYOR_1_ANIO         0
SANCION_MAYOR_3_ANIOS        0
LOG_DURACION_DIAS            0
ES_PERSONA_NATURAL           0
ES_PERSONA_JURIDICA          0
ES_OTRO_TIPO_RUC             0
MOTIVO_DOC_FALSA             0
MOTIVO_INFO_INEXACTA         0
MOTIVO_INCUMPLIMIENTO        0
MOTIVO_CONTRATAR_IMPEDIDO    0
NUM_MOTIVOS_INFRACCION       0
TIENE_MULTIPLES_MOTIVOS      0
motivo_grupo                 0
duracion_grupo               0
dtype: int64


In [5]:
# ------------------------------------------
# 5. Codificación de variables categóricas
# ------------------------------------------

X = pd.get_dummies(X, drop_first=True)

print("\nDimensión de X después de get_dummies:")
print(X.shape)

print("\nPrimeras columnas del dataset modelable:")
print(X.columns.tolist()[:30])


Dimensión de X después de get_dummies:
(3875, 26)

Primeras columnas del dataset modelable:
['ANIO_INICIO_SANCION', 'MES_INICIO_SANCION', 'TRIMESTRE_INICIO_SANCION', 'DURACION_MESES', 'DURACION_ANIOS', 'SANCION_MAYOR_1_ANIO', 'SANCION_MAYOR_3_ANIOS', 'LOG_DURACION_DIAS', 'ES_PERSONA_NATURAL', 'ES_PERSONA_JURIDICA', 'ES_OTRO_TIPO_RUC', 'MOTIVO_DOC_FALSA', 'MOTIVO_INFO_INEXACTA', 'MOTIVO_INCUMPLIMIENTO', 'MOTIVO_CONTRATAR_IMPEDIDO', 'NUM_MOTIVOS_INFRACCION', 'TIENE_MULTIPLES_MOTIVOS', 'motivo_grupo_Impedimento para contratar', 'motivo_grupo_No perfecciona/suscribe contrato', 'motivo_grupo_Otros', 'motivo_grupo_Resolución/rescisión contractual', 'duracion_grupo_6-12 meses', 'duracion_grupo_Fecha fin < inicio', 'duracion_grupo_Hasta 6 meses', 'duracion_grupo_Más de 3 años', 'duracion_grupo_Sin fecha fin']


In [6]:
# ------------------------------------------
# 6. División train/test con estratificación
# ------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nDimensión de X_train:", X_train.shape)
print("Dimensión de X_test:", X_test.shape)
print("Dimensión de y_train:", y_train.shape)
print("Dimensión de y_test:", y_test.shape)

print("\nDistribución en y_train:")
print(y_train.value_counts(normalize=True).round(4) * 100)

print("\nDistribución en y_test:")
print(y_test.value_counts(normalize=True).round(4) * 100)


Dimensión de X_train: (3100, 26)
Dimensión de X_test: (775, 26)
Dimensión de y_train: (3100,)
Dimensión de y_test: (775,)

Distribución en y_train:
target_sancionado_24m
0    81.0
1    19.0
Name: proportion, dtype: float64

Distribución en y_test:
target_sancionado_24m
0    81.03
1    18.97
Name: proportion, dtype: float64


In [7]:
# ------------------------------------------
# 7. Entrenar el modelo final seleccionado
# ------------------------------------------

modelo_final = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

modelo_final.fit(X_train, y_train)

print("Modelo final entrenado correctamente.")
print("Clases detectadas:")
print(modelo_final.classes_)

Modelo final entrenado correctamente.
Clases detectadas:
[0 1]


In [8]:
# ------------------------------------------
# 8. Evaluación del modelo antes de guardar
# ------------------------------------------

y_pred = modelo_final.predict(X_test)
y_prob = modelo_final.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
matriz = confusion_matrix(y_test, y_pred)
reporte = classification_report(y_test, y_pred)

precision_1 = precision_score(y_test, y_pred, pos_label=1)
recall_1 = recall_score(y_test, y_pred, pos_label=1)
f1_1 = f1_score(y_test, y_pred, pos_label=1)
roc_auc = roc_auc_score(y_test, y_prob)

print("\n" + "="*60)
print("RESULTADOS DEL MODELO FINAL ANTES DEL GUARDADO")
print("="*60)

print(f"\nAccuracy del modelo final: {accuracy:.4f}")

print("\nMatriz de confusión:")
print(matriz)

print("\nReporte de clasificación:")
print(reporte)

print("\nMétricas clase positiva 1:")
print(f"Precision clase 1: {precision_1:.4f}")
print(f"Recall clase 1:    {recall_1:.4f}")
print(f"F1-score clase 1:  {f1_1:.4f}")
print(f"ROC-AUC:           {roc_auc:.4f}")


RESULTADOS DEL MODELO FINAL ANTES DEL GUARDADO

Accuracy del modelo final: 0.7071

Matriz de confusión:
[[490 138]
 [ 89  58]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.85      0.78      0.81       628
           1       0.30      0.39      0.34       147

    accuracy                           0.71       775
   macro avg       0.57      0.59      0.58       775
weighted avg       0.74      0.71      0.72       775


Métricas clase positiva 1:
Precision clase 1: 0.2959
Recall clase 1:    0.3946
F1-score clase 1:  0.3382
ROC-AUC:           0.6579


In [9]:
# ------------------------------------------
# 9. Guardar modelo en formato .joblib
# ------------------------------------------

joblib.dump(modelo_final, ruta_joblib, compress=3)

# ------------------------------------------
# 10. Guardar modelo en formato .pkl
# ------------------------------------------

with open(ruta_pkl, "wb") as archivo_pkl:
    pickle.dump(modelo_final, archivo_pkl)

# ------------------------------------------
# 11. Guardar columnas de entrenamiento
# ------------------------------------------

columnas_entrenamiento = pd.DataFrame({
    "columna": X_train.columns.tolist()
})

columnas_entrenamiento.to_csv(
    ruta_columnas,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------
# 12. Crear artefacto adicional con metadatos
# ------------------------------------------

artefacto_final = {
    "nombre_proyecto": "RiskScore OSCE",
    "descripcion": "Modelo predictivo binario para estimar riesgo de sanción futura de proveedores.",
    "modelo": modelo_final,
    "tipo_modelo": "RandomForestClassifier",
    "target": target,
    "interpretacion_target": {
        "0": "Proveedor sin sanción posterior dentro de 24 meses",
        "1": "Proveedor con sanción posterior dentro de 24 meses"
    },
    "columnas_originales_modelo": columnas_existentes,
    "columnas_entrenamiento": X_train.columns.tolist(),
    "columnas_numericas": columnas_numericas,
    "columnas_categoricas": columnas_categoricas,
    "medianas_numericas": medianas_numericas,
    "parametros_modelo": modelo_final.get_params(),
    "clases_modelo": modelo_final.classes_.tolist(),
    "metricas_validacion": {
        "accuracy": accuracy,
        "precision_clase_1": precision_1,
        "recall_clase_1": recall_1,
        "f1_clase_1": f1_1,
        "roc_auc": roc_auc
    }
}

joblib.dump(artefacto_final, ruta_artefacto, compress=3)

print("\nArchivos guardados correctamente:")
print("- Modelo .joblib:", ruta_joblib)
print("- Modelo .pkl:", ruta_pkl)
print("- Artefacto completo:", ruta_artefacto)
print("- Columnas de entrenamiento:", ruta_columnas)


Archivos guardados correctamente:
- Modelo .joblib: C:\Proyecto-Risk-Score-OECE\models\riskscore_osce_random_forest.joblib
- Modelo .pkl: C:\Proyecto-Risk-Score-OECE\models\riskscore_osce_random_forest.pkl
- Artefacto completo: C:\Proyecto-Risk-Score-OECE\models\artefacto_riskscore_osce_random_forest.joblib
- Columnas de entrenamiento: C:\Proyecto-Risk-Score-OECE\models\columnas_entrenamiento_riskscore_osce.csv


In [10]:
# ------------------------------------------
# 13. Verificación del archivo .joblib
# ------------------------------------------

modelo_joblib_cargado = joblib.load(ruta_joblib)

y_pred_joblib = modelo_joblib_cargado.predict(X_test)

accuracy_joblib = accuracy_score(y_test, y_pred_joblib)
matriz_joblib = confusion_matrix(y_test, y_pred_joblib)
reporte_joblib = classification_report(y_test, y_pred_joblib)

print("\n" + "="*60)
print("VERIFICACIÓN DEL MODELO CARGADO DESDE .JOBLIB")
print("="*60)

print(f"\nAccuracy de verificación (.joblib): {accuracy_joblib:.4f}")

print("\nMatriz de confusión (.joblib):")
print(matriz_joblib)

print("\nReporte de clasificación (.joblib):")
print(reporte_joblib)


VERIFICACIÓN DEL MODELO CARGADO DESDE .JOBLIB

Accuracy de verificación (.joblib): 0.7071

Matriz de confusión (.joblib):
[[490 138]
 [ 89  58]]

Reporte de clasificación (.joblib):
              precision    recall  f1-score   support

           0       0.85      0.78      0.81       628
           1       0.30      0.39      0.34       147

    accuracy                           0.71       775
   macro avg       0.57      0.59      0.58       775
weighted avg       0.74      0.71      0.72       775



In [11]:
# ------------------------------------------
# 14. Verificación del archivo .pkl
# ------------------------------------------

with open(ruta_pkl, "rb") as archivo_pkl:
    modelo_pkl_cargado = pickle.load(archivo_pkl)

y_pred_pkl = modelo_pkl_cargado.predict(X_test)

accuracy_pkl = accuracy_score(y_test, y_pred_pkl)
matriz_pkl = confusion_matrix(y_test, y_pred_pkl)
reporte_pkl = classification_report(y_test, y_pred_pkl)

print("\n" + "="*60)
print("VERIFICACIÓN DEL MODELO CARGADO DESDE .PKL")
print("="*60)

print(f"\nAccuracy de verificación (.pkl): {accuracy_pkl:.4f}")

print("\nMatriz de confusión (.pkl):")
print(matriz_pkl)

print("\nReporte de clasificación (.pkl):")
print(reporte_pkl)


VERIFICACIÓN DEL MODELO CARGADO DESDE .PKL

Accuracy de verificación (.pkl): 0.7071

Matriz de confusión (.pkl):
[[490 138]
 [ 89  58]]

Reporte de clasificación (.pkl):
              precision    recall  f1-score   support

           0       0.85      0.78      0.81       628
           1       0.30      0.39      0.34       147

    accuracy                           0.71       775
   macro avg       0.57      0.59      0.58       775
weighted avg       0.74      0.71      0.72       775



In [12]:
# ------------------------------------------
# 15. Verificación del artefacto completo
# ------------------------------------------

artefacto_cargado = joblib.load(ruta_artefacto)

print("Artefacto cargado correctamente.")
print("\nNombre del proyecto:")
print(artefacto_cargado["nombre_proyecto"])

print("\nTarget:")
print(artefacto_cargado["target"])

print("\nClases del modelo:")
print(artefacto_cargado["clases_modelo"])

print("\nCantidad de columnas de entrenamiento:")
print(len(artefacto_cargado["columnas_entrenamiento"]))

print("\nMétricas guardadas:")
print(artefacto_cargado["metricas_validacion"])

Artefacto cargado correctamente.

Nombre del proyecto:
RiskScore OSCE

Target:
target_sancionado_24m

Clases del modelo:
[0, 1]

Cantidad de columnas de entrenamiento:
26

Métricas guardadas:
{'accuracy': 0.7070967741935484, 'precision_clase_1': 0.29591836734693877, 'recall_clase_1': 0.3945578231292517, 'f1_clase_1': 0.33819241982507287, 'roc_auc': 0.6578762078079639}


In [13]:
# ------------------------------------------
# FUNCIÓN PARA PREPARAR NUEVOS DATOS
# ------------------------------------------

def preparar_nuevos_datos(df_nuevo, artefacto):
    """
    Prepara nuevos datos para que tengan la misma estructura
    que el dataset usado en entrenamiento.
    """

    columnas_originales = artefacto["columnas_originales_modelo"]
    columnas_entrenamiento = artefacto["columnas_entrenamiento"]
    columnas_numericas = artefacto["columnas_numericas"]
    columnas_categoricas = artefacto["columnas_categoricas"]
    medianas_numericas = artefacto["medianas_numericas"]

    df = df_nuevo.copy()

    # Seleccionar columnas originales usadas por el modelo
    df = df[columnas_originales].copy()

    # Reemplazar infinitos
    df = df.replace([np.inf, -np.inf], np.nan)

    # Imputar numéricas
    for col in columnas_numericas:
        if col in df.columns:
            df[col] = df[col].fillna(medianas_numericas.get(col, 0))

    # Imputar categóricas
    for col in columnas_categoricas:
        if col in df.columns:
            df[col] = df[col].fillna("SIN_DATO")

    # Codificar categóricas
    df = pd.get_dummies(df, drop_first=True)

    # Alinear columnas con entrenamiento
    df = df.reindex(columns=columnas_entrenamiento, fill_value=0)

    return df